## Setting the env


# INITIAL IMPORTS FORM NEEDED PACKAGES

In [1]:
from pathlib import Path
import geopandas as gpd
import osmium

Reading the files and checking the size of the bpf history file

In [2]:
DATA_DIR = Path("data")
OSH_PATH = DATA_DIR / "history_file.osh.pbf"
AOI_PATH = DATA_DIR / "aoi.geojson"   # or .shp, .gpkg

print(f"{OSH_PATH.stat().st_size / 1e6:.1f} MB")

141.1 MB


Read the AOI and set the projection

In [3]:
aoi = gpd.read_file(AOI_PATH)

print(aoi.crs, len(aoi), aoi.geom_type.unique())

if aoi.crs is None:
    raise ValueError("AOI has no CRS: set it explicitly, don't guess")
aoi = aoi.to_crs(4326)


EPSG:4326 1 <ArrowStringArray>
['MultiPolygon']
Length: 1, dtype: str


Inspect PBF file

In [4]:
reader = osmium.io.Reader(str(OSH_PATH), osmium.osm.osm_entity_bits.NOTHING)
header = reader.header()

print("History file:", header.has_multiple_object_versions)
print("Bounding box:", header.box())
print("Generator:", header.get("generator"))
print("Replication timestamp:", header.get("osmosis_replication_timestamp"))
reader.close()

History file: True
Bounding box: (35.2770970/32.3066000 42.3838300/37.3223940)
Generator: osmium/1.16.0
Replication timestamp: 


4. Stream the history file and describe it

In [ ]:
import time
from collections import Counter
import osmium

class HistorySummary(osmium.SimpleHandler): ##Defining subclass because pyosmium requires it.
    def __init__(self): 
        super().__init__()
        self.versions = Counter()        # all versions, per type
        self.objects = Counter()         # distinct objects, per type
        self.deletions = Counter()       # versions where the object was deleted
        self.edits_per_year = Counter()
        self.users = set()
        self.changesets = set()
        self.first_ts = None
        self.last_ts = None
        self._last_id = {}               # last ID seen, per type

    def _process(self, otype, o):
        # Distinct-object count, relying on (type, id, version) sort order
        prev = self._last_id.get(otype)
        if prev is not None and o.id < prev:
            raise RuntimeError(f"File not sorted: {otype} {o.id} after {prev}")
        if o.id != prev:
            self.objects[otype] += 1
            self._last_id[otype] = o.id

        self.versions[otype] += 1
        if not o.visible:
            self.deletions[otype] += 1

        ts = o.timestamp
        if self.first_ts is None or ts < self.first_ts:
            self.first_ts = ts
        if self.last_ts is None or ts > self.last_ts:
            self.last_ts = ts
        self.edits_per_year[ts.year] += 1

        self.users.add(o.uid)
        self.changesets.add(o.changeset)

    def node(self, n):
        self._process("node", n)

    def way(self, w):
        self._process("way", w)

    def relation(self, r):
        self._process("relation", r)


summary = HistorySummary()
t0 = time.perf_counter()
summary.apply_file(str(OSH_PATH))
print(f"Done in {time.perf_counter() - t0:.1f} s")

Done in 154.6 s


In [ ]:
print("First edit:", summary.first_ts)
print("Last edit: ", summary.last_ts)
print()
for t in ("node", "way", "relation"):
    print(f"{t:9} objects={summary.objects[t]:>10,}  "
          f"versions={summary.versions[t]:>10,}  "
          f"deletions={summary.deletions[t]:>9,}")
print()
print("Distinct users: ", f"{len(summary.users):,}")
print("Distinct changesets:", f"{len(summary.changesets):,}")
print()
print(dict(sorted(summary.edits_per_year.items())))

First edit: 2006-05-15 01:04:26+00:00
Last edit:  2026-09-20 23:12:15+00:00

node      objects=13,900,091  versions=17,694,724  deletions=  964,630
way       objects= 1,986,632  versions= 2,627,127  deletions=  119,553
relation  objects=     4,187  versions=    56,215  deletions=      753

Distinct users:      25,045
Distinct changesets: 251,091

{2006: 171, 2007: 27232, 2008: 152047, 2009: 209031, 2010: 83833, 2011: 309493, 2012: 711170, 2013: 571949, 2014: 404166, 2015: 605654, 2016: 696982, 2017: 664121, 2018: 309627, 2019: 555433, 2020: 575386, 2021: 1693430, 2022: 527247, 2023: 8143854, 2024: 908711, 2025: 1398886, 2026: 1829643}


5. Preparing AOI to clip the pbf file with AOI geometry

In [ ]:
import numpy as np
import shapely

aoi_geom = aoi.union_all()          # one MultiPolygon geometry
shapely.prepare(aoi_geom)           # Create spatial index to speeds up processes 
minx, miny, maxx, maxy = aoi_geom.bounds
print(aoi_geom.bounds)

(35.896452, 32.72490980936, 36.177534086, 33.41195703719)


In [10]:
ids, lons, lats = [], [], []

t0 = time.perf_counter()
for n in osmium.FileProcessor(str(OSH_PATH), osmium.osm.NODE):
    if not n.visible:               # deleted versions have no location
        continue
    loc = n.location
    if not loc.valid():
        continue
    lon, lat = loc.lon, loc.lat
    if minx <= lon <= maxx and miny <= lat <= maxy:
        ids.append(n.id)
        lons.append(lon)
        lats.append(lat)
print(f"{len(ids):,} node versions in bbox, {time.perf_counter() - t0:.0f} s")

180,974 node versions in bbox, 162 s


In [11]:
ids = np.asarray(ids, dtype=np.int64)
inside = shapely.intersects_xy(aoi_geom, np.asarray(lons), np.asarray(lats))

node_ids = set(np.unique(ids[inside]).tolist())
print(f"{inside.sum():,} node versions inside AOI")
print(f"{len(node_ids):,} distinct nodes selected")

del lons, lats, inside              # free memory, we only need the IDs

180,973 node versions inside AOI
160,236 distinct nodes selected


5b. Select the ways

In [12]:
way_ids = set()

t0 = time.perf_counter()
for w in osmium.FileProcessor(str(OSH_PATH), osmium.osm.WAY):
    if any(nd.ref in node_ids for nd in w.nodes):
        way_ids.add(w.id)
print(f"{len(way_ids):,} ways selected, {time.perf_counter() - t0:.0f} s")

19,676 ways selected, 90 s


complete the ways

In [13]:
extra_node_ids = set()

t0 = time.perf_counter()
for w in osmium.FileProcessor(str(OSH_PATH), osmium.osm.WAY):
    if w.id in way_ids:
        extra_node_ids.update(nd.ref for nd in w.nodes)

extra_node_ids -= node_ids          # keep only the new ones
print(f"{len(extra_node_ids):,} extra nodes to complete ways, "
      f"{time.perf_counter() - t0:.0f} s")

16,147 extra nodes to complete ways, 24 s
